# 09 - Fine-tune Whisper-Arabic + Fine-Tashkeel Pipeline

**Architecture:** Two-stage pipeline
1. **Stage 1**: Whisper Arabic → Undiacritized transcription
2. **Stage 2**: Fine-Tashkeel → Diacritized text

## Why This Approach?
- **Whisper**: Best ASR for Arabic (state-of-the-art)
- **Fine-Tashkeel**: Best text diacritization
- **Modular**: Can fine-tune each component separately

## Training Plan:
1. Fine-tune Whisper on KSAA audio data
2. Use existing Fine-Tashkeel (already trained)
3. Joint inference: Audio → Whisper → Fine-Tashkeel → Diacritized

## Tasks:
1. Fine-tune Whisper on KSAA Arabic audio
2. Build inference pipeline
3. Evaluate on dev set
4. Generate test submission

In [1]:
!pip install -q transformers torch accelerate datasets tqdm librosa soundfile pandas

In [2]:
# Setup
import os, sys, json, re, torch, gc, zipfile
from datetime import datetime
from tqdm import tqdm
from pathlib import Path
import pandas as pd
import librosa
import numpy as np
import warnings
warnings.filterwarnings('ignore')

IS_RUNPOD = False  # Cloud detection removed
PROJECT_DIR = Path('.')
sys.path.insert(0, str(PROJECT_DIR))
sys.path.insert(0, str(PROJECT_DIR / 'notebooks'))

# Import utilities (includes safeguards for empty responses & separated characters)
from utils_diacritization import (
    normalize_arabic, remove_diacritics, postprocess_diacritics, compute_metrics,
    is_valid_output, repair_output, get_safe_seq2seq_generation_config
)

DATA_DIR = PROJECT_DIR / 'Public_Data_TrainDev'
TRAIN_DIR = DATA_DIR / 'Train'
TRAIN_AUDIO_DIR = TRAIN_DIR / 'train_audio'
TRAIN_TEXT_DIR = TRAIN_DIR / 'train_text'
DEV_INPUT = DATA_DIR / 'dev input-output' / 'Dev_input.json'
DEV_OUTPUT = DATA_DIR / 'dev input-output' / 'Dev_output.json'
DEV_AUDIO_DIR = DATA_DIR / 'Dev' / 'dev_audio'
TEST_DIR = PROJECT_DIR / 'Test'
TEST_INPUT = TEST_DIR / 'test_input.json'
TEST_AUDIO_DIR = TEST_DIR / 'test_audio'
OUTPUT_DIR = PROJECT_DIR / 'outputs'
SUBMISSION_DIR = PROJECT_DIR / 'submissions'
CHECKPOINTS_DIR = PROJECT_DIR / 'checkpoints' / 'whisper_tashkeel_ft'

for d in [OUTPUT_DIR, SUBMISSION_DIR, CHECKPOINTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Environment: 'Local' | Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB)")

def clear_gpu():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        gc.collect()

Environment: Local | Device: cuda
GPU: NVIDIA RTX A5000 (23.6 GB)


## 1. Load Training Data

In [3]:
train_tsv = TRAIN_DIR / 'train_list.tsv'
train_df = pd.read_csv(train_tsv, sep='\t')
print(f"Training samples: {len(train_df)}")

def load_audio(path, sr=16000):
    try:
        audio, _ = librosa.load(path, sr=sr)
        return audio
    except:
        return None

def load_training_data(max_samples=None):
    data = []
    
    for idx, row in tqdm(train_df.iterrows(), total=len(train_df), desc="Loading data"):
        if max_samples and len(data) >= max_samples:
            break
            
        stem = row['stem']
        audio_path = TRAIN_AUDIO_DIR / f"{stem}.mp3"
        text_path = TRAIN_TEXT_DIR / f"{stem}.txt"
        
        if audio_path.exists() and text_path.exists():
            with open(text_path, 'r', encoding='utf-8') as f:
                diacritized = normalize_arabic(f.read().strip())
            
            # For Whisper training, we need undiacritized text
            undiacritized = remove_diacritics(diacritized)
            
            data.append({
                'id': stem,
                'audio_path': str(audio_path),
                'text_undiacritized': undiacritized,
                'text_diacritized': diacritized
            })
    
    return data

train_data = load_training_data()
print(f"Loaded {len(train_data)} samples")

Training samples: 2327


Loading data: 100%|██████████| 2327/2327 [00:11<00:00, 206.38it/s]

Loaded 2327 samples


## 2. Load Whisper Model for Fine-tuning

In [4]:
from transformers import (
    WhisperProcessor, WhisperForConditionalGeneration,
    Seq2SeqTrainer, Seq2SeqTrainingArguments
)
from datasets import Dataset, Audio

# Use Whisper Small for efficiency (or medium for better accuracy)
WHISPER_MODEL = 'openai/whisper-small'
MODEL_KEY = 'whisper_tashkeel_ft'

print(f"Loading {WHISPER_MODEL}...")
whisper_processor = WhisperProcessor.from_pretrained(WHISPER_MODEL)
whisper_model = WhisperForConditionalGeneration.from_pretrained(
    WHISPER_MODEL,
    torch_dtype=torch.float32,  # Full precision for training stability
).to(device)

# Set language to Arabic
whisper_model.config.forced_decoder_ids = whisper_processor.get_decoder_prompt_ids(
    language="ar", task="transcribe"
)

# NOTE: Do NOT enable gradient_checkpointing for Whisper - causes tensor mismatch errors
# whisper_model.gradient_checkpointing_enable()  # DISABLED

print(f"Whisper loaded: {sum(p.numel() for p in whisper_model.parameters()):,} params")

Loading openai/whisper-small...


preprocessor_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

Whisper loaded: 241,734,912 params


## 3. Prepare Dataset for Whisper Training

In [5]:
from dataclasses import dataclass
from typing import Any, Dict, List, Union

MAX_AUDIO_LENGTH = 30 * 16000  # 30 seconds

def prepare_dataset(batch):
    """Prepare audio features and labels for Whisper."""
    audio_arrays = []
    texts = []
    
    for audio_path, text in zip(batch['audio_path'], batch['text_undiacritized']):
        audio = load_audio(audio_path)
        if audio is None:
            audio = np.zeros(16000)
        if len(audio) > MAX_AUDIO_LENGTH:
            audio = audio[:MAX_AUDIO_LENGTH]
        audio_arrays.append(audio)
        texts.append(text)
    
    # Process audio to mel spectrogram
    features = whisper_processor(
        audio_arrays,
        sampling_rate=16000,
        return_tensors="np"
    )
    
    # Tokenize text
    labels = whisper_processor.tokenizer(
        texts,
        return_tensors="np",
        padding=True,
        truncation=True,
        max_length=448
    )
    
    return {
        'input_features': features.input_features.tolist(),
        'labels': labels.input_ids.tolist()
    }

# Create dataset
train_dataset = Dataset.from_list(train_data)
train_dataset = train_dataset.map(
    prepare_dataset,
    batched=True,
    batch_size=8,
    remove_columns=['id', 'audio_path', 'text_undiacritized', 'text_diacritized'],
    desc="Processing audio"
)

print(f"Dataset ready: {len(train_dataset)} samples")

Processing audio:   0%|          | 0/2327 [00:00<?, ? examples/s]

[src/libmpg123/id3.c:process_comment():587] error: No comment text / valid description?


Dataset ready: 2327 samples


In [6]:
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    
    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # Stack input features
        input_features = torch.tensor([f['input_features'] for f in features])
        
        # Pad labels
        label_features = [{'input_ids': f['labels']} for f in features]
        labels_batch = self.processor.tokenizer.pad(
            label_features,
            return_tensors='pt'
        )
        
        # Replace padding with -100
        labels = labels_batch['input_ids'].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        
        return {
            'input_features': input_features,
            'labels': labels
        }

data_collator = DataCollatorSpeechSeq2SeqWithPadding(whisper_processor)

## 4. Train Whisper

In [11]:
# Training args for Whisper - Optimized for 24GB GPU
# NOTE: FP16 disabled to avoid gradient checkpointing issues
training_args = Seq2SeqTrainingArguments(
    output_dir=str(CHECKPOINTS_DIR / 'whisper'),
    
    # Training - Whisper Small (244M) - reduced batch for stability
    num_train_epochs=3,
    per_device_train_batch_size=4,   # Reduced for FP32 training
    gradient_accumulation_steps=4,   # Effective batch: 16
    
    # Optimizer
    learning_rate=1e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    
    # Precision - Disable FP16 to avoid checkpoint errors
    fp16=False,
    bf16=False,
    
    # Saving
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
    
    # Logging
    logging_steps=50,
    report_to="none",
    
    # Generation
    predict_with_generate=True,
    generation_max_length=225,
    
    # Performance
    dataloader_num_workers=4,
    dataloader_pin_memory=True,
)

# Use processing_class instead of tokenizer (newer transformers API)
trainer = Seq2SeqTrainer(
    model=whisper_model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=data_collator,
    processing_class=whisper_processor.feature_extractor,
)

print("Whisper trainer ready")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Batch size: {training_args.per_device_train_batch_size}")
print(f"  Effective batch: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"  Precision: FP32 (for stability)")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Whisper trainer ready
  Epochs: 3
  Batch size: 4
  Effective batch: 16
  Precision: FP32 (for stability)


In [4]:
clear_gpu()

In [12]:
# Train Whisper
clear_gpu()

checkpoint = None
whisper_checkpoints = list((CHECKPOINTS_DIR / 'whisper').glob('checkpoint-*')) if (CHECKPOINTS_DIR / 'whisper').exists() else []
if whisper_checkpoints:
    checkpoint = str(max(whisper_checkpoints, key=lambda x: int(x.name.split('-')[1])))
    print(f"Resuming from: {checkpoint}")

print("Training Whisper...")
trainer.train(resume_from_checkpoint=checkpoint)

Training Whisper...


Step,Training Loss
50,5.362966
100,3.642563
150,2.227786
200,1.271744
250,1.110994
300,0.974901
350,0.707981
400,0.741299


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=438, training_loss=1.8928684774599118, metrics={'train_runtime': 647.6472, 'train_samples_per_second': 10.779, 'train_steps_per_second': 0.676, 'total_flos': 2.01461467963392e+18, 'train_loss': 1.8928684774599118, 'epoch': 3.0})

In [13]:
# Save Whisper
whisper_final_path = CHECKPOINTS_DIR / 'whisper' / 'final'
trainer.save_model(str(whisper_final_path))
whisper_processor.save_pretrained(str(whisper_final_path))
print(f"Whisper saved to: {whisper_final_path}")
clear_gpu()

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Whisper saved to: ./checkpoints/whisper_tashkeel_ft/whisper/final


## 5. Load Fine-Tashkeel for Pipeline

In [14]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

TASHKEEL_MODEL = 'basharalrfooh/Fine-Tashkeel'

# Clear Whisper training model first to make room
print("Clearing Whisper training model...")
del whisper_model
clear_gpu()

print(f"Loading {TASHKEEL_MODEL} for diacritization...")
tashkeel_tokenizer = AutoTokenizer.from_pretrained(TASHKEEL_MODEL)
tashkeel_model = AutoModelForSeq2SeqLM.from_pretrained(
    TASHKEEL_MODEL,
    torch_dtype=torch.float16,  # FP16 for speed
    device_map="auto"
)
tashkeel_model.eval()

print(f"Fine-Tashkeel loaded (FP16): {sum(p.numel() for p in tashkeel_model.parameters()):,} params")

Clearing Whisper training model...
Loading basharalrfooh/Fine-Tashkeel for diacritization...


config.json:   0%|          | 0.00/798 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


pytorch_model.bin:   0%|          | 0.00/2.88G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/308 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/164 [00:00<?, ?B/s]

Fine-Tashkeel loaded (FP16): 720,870,144 params


## 6. Load Fine-tuned Whisper for Inference

In [15]:
# Load fine-tuned Whisper for inference (FP16 for speed)
whisper_final_path = CHECKPOINTS_DIR / 'whisper' / 'final'
if whisper_final_path.exists():
    print(f"Loading fine-tuned Whisper from {whisper_final_path}")
    whisper_model = WhisperForConditionalGeneration.from_pretrained(
        whisper_final_path,
        torch_dtype=torch.float16  # FP16 for faster inference
    ).to(device)
    whisper_processor = WhisperProcessor.from_pretrained(whisper_final_path)
else:
    print("Loading base Whisper model")
    whisper_model = WhisperForConditionalGeneration.from_pretrained(
        'openai/whisper-small',
        torch_dtype=torch.float16
    ).to(device)
    whisper_processor = WhisperProcessor.from_pretrained('openai/whisper-small')

whisper_model.eval()
print("Pipeline ready (both models in FP16)")

Loading fine-tuned Whisper from ./checkpoints/whisper_tashkeel_ft/whisper/final


model.safetensors:   0%|          | 0.00/2.88G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Pipeline ready (both models in FP16)


In [16]:
@torch.inference_mode()
def transcribe_and_diacritize(audio_path: str, fallback_text: str = "") -> str:
    """
    Two-stage pipeline with safeguards:
    1. Whisper: Audio → Undiacritized text
    2. Fine-Tashkeel: Undiacritized → Diacritized
    """
    # Stage 1: ASR with Whisper
    audio = load_audio(audio_path)
    if audio is None:
        transcription = fallback_text
    else:
        if len(audio) > MAX_AUDIO_LENGTH:
            audio = audio[:MAX_AUDIO_LENGTH]
        
        input_features = whisper_processor(
            audio, sampling_rate=16000, return_tensors="pt"
        ).input_features.to(device)
        
        if device == "cuda":
            input_features = input_features.half()
        
        generated_ids = whisper_model.generate(
            input_features,
            language="ar",
            task="transcribe"
        )
        
        transcription = whisper_processor.batch_decode(
            generated_ids, skip_special_tokens=True
        )[0].strip()
    
    # Fallback if transcription is empty
    if not transcription:
        transcription = fallback_text
    
    # Stage 2: Diacritization with Fine-Tashkeel
    transcription = normalize_arabic(transcription)
    
    inputs = tashkeel_tokenizer(
        transcription,
        return_tensors="pt",
        max_length=512,
        truncation=True
    )
    inputs = {k: v.to(tashkeel_model.device) for k, v in inputs.items()}
    
    # Get safe generation config
    gen_config = get_safe_seq2seq_generation_config()
    gen_config['max_length'] = 512
    
    outputs = tashkeel_model.generate(
        **inputs,
        **gen_config
    )
    
    diacritized = tashkeel_tokenizer.decode(outputs[0], skip_special_tokens=True).strip()
    
    # Repair output (validates and fixes issues, fallback to input text if invalid)
    diacritized = repair_output(diacritized, fallback_text, fallback_to_input=True)
    
    return diacritized

## 7. Evaluate on Dev Set

In [17]:
with open(DEV_INPUT, 'r', encoding='utf-8') as f:
    dev_input = json.load(f)
with open(DEV_OUTPUT, 'r', encoding='utf-8') as f:
    dev_output = json.load(f)

CHECKPOINT_FILE = OUTPUT_DIR / f"{MODEL_KEY}_checkpoint.json"

def load_checkpoint():
    if CHECKPOINT_FILE.exists():
        with open(CHECKPOINT_FILE, 'r', encoding='utf-8') as f:
            data = json.load(f)
            return set(data.get('processed_ids', [])), data.get('predictions', [])
    return set(), []

def save_checkpoint(predictions):
    with open(CHECKPOINT_FILE, 'w', encoding='utf-8') as f:
        json.dump({'processed_ids': [p['id'] for p in predictions], 'predictions': predictions}, f, ensure_ascii=False)

processed_ids, dev_predictions = load_checkpoint()
print(f"Dev: {len(dev_input)} samples, {len(processed_ids)} already done")

for item in tqdm(dev_input, desc="Dev Set"):
    if item['id'] in processed_ids:
        continue
    try:
        audio_path = DEV_AUDIO_DIR / f"{item['id']}.mp3"
        result = transcribe_and_diacritize(str(audio_path), item['text_undiacritized'])
        dev_predictions.append({'id': item['id'], 'text_diacritized': result})
        save_checkpoint(dev_predictions)
    except Exception as e:
        print(f"Error: {e}")
        dev_predictions.append({'id': item['id'], 'text_diacritized': item['text_undiacritized']})
        save_checkpoint(dev_predictions)

with open(OUTPUT_DIR / f"{MODEL_KEY}_dev_predictions.json", 'w', encoding='utf-8') as f:
    json.dump(dev_predictions, f, ensure_ascii=False, indent=2)

Dev: 260 samples, 0 already done


Dev Set:   0%|          | 0/260 [00:00<?, ?it/s]The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transf

In [18]:
# Compute metrics
print("\n" + "="*60)
print("DEV SET RESULTS (Whisper + Fine-Tashkeel Pipeline)")
print("="*60)

metrics = compute_metrics(dev_predictions, dev_output, exclude_irab=False)
print(f"\n[Including I'rab]")
print(f"  DER: {metrics['DER']*100:.2f}%")
print(f"  WER: {metrics['WER']*100:.2f}% (PRIMARY)")
print(f"  SER: {metrics['SER']*100:.2f}%")


DEV SET RESULTS (Whisper + Fine-Tashkeel Pipeline)

[Including I'rab]
  DER: 55.05%
  WER: 77.40% (PRIMARY)
  SER: 99.62%


## 8. Generate Test Submission

In [19]:
with open(TEST_INPUT, 'r', encoding='utf-8') as f:
    test_input = json.load(f)

TEST_CHECKPOINT_FILE = OUTPUT_DIR / f"{MODEL_KEY}_test_checkpoint.json"

def load_test_checkpoint():
    if TEST_CHECKPOINT_FILE.exists():
        with open(TEST_CHECKPOINT_FILE, 'r', encoding='utf-8') as f:
            data = json.load(f)
            return set(data.get('processed_ids', [])), data.get('predictions', [])
    return set(), []

def save_test_checkpoint(predictions):
    with open(TEST_CHECKPOINT_FILE, 'w', encoding='utf-8') as f:
        json.dump({'processed_ids': [p['id'] for p in predictions], 'predictions': predictions}, f, ensure_ascii=False)

test_processed_ids, test_predictions = load_test_checkpoint()
print(f"Test: {len(test_input)} samples, {len(test_processed_ids)} already done")

for item in tqdm(test_input, desc="Test Set"):
    if item['id'] in test_processed_ids:
        continue
    try:
        audio_path = TEST_AUDIO_DIR / f"{item['id']}.mp3"
        result = transcribe_and_diacritize(str(audio_path), item['text_undiacritized'])
        test_predictions.append({'id': item['id'], 'text_diacritized': result})
        save_test_checkpoint(test_predictions)
    except:
        test_predictions.append({'id': item['id'], 'text_diacritized': item['text_undiacritized']})
        save_test_checkpoint(test_predictions)

with open(OUTPUT_DIR / f"{MODEL_KEY}_test_predictions.json", 'w', encoding='utf-8') as f:
    json.dump(test_predictions, f, ensure_ascii=False, indent=2)

Test: 328 samples, 0 already done


Test Set: 100%|██████████| 328/328 [16:08<00:00,  2.95s/it]


In [20]:
# Create submission
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

json_file = SUBMISSION_DIR / f"{MODEL_KEY}_submission.json"
with open(json_file, 'w', encoding='utf-8') as f:
    json.dump(test_predictions, f, ensure_ascii=False, indent=2)

zip_file = SUBMISSION_DIR / f"{MODEL_KEY}_submission_{timestamp}.zip"
with zipfile.ZipFile(zip_file, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(json_file, 'submission.json')

print("="*60)
print("SUBMISSION READY")
print("="*60)
print(f"ZIP: {zip_file}")

SUBMISSION READY
ZIP: ./submissions/whisper_tashkeel_ft_submission_20260224_085300.zip


In [21]:
# Sync and cleanup
import subprocess
sync_script = PROJECT_DIR / 'sync_results.sh'
if sync_script.exists() and False:
    print("Syncing to Google Drive...")
    subprocess.run(['bash', str(sync_script)])

# Final cleanup - free GPU memory
print("\nCleaning up GPU memory...")
del whisper_model
del whisper_processor
del tashkeel_model
del tashkeel_tokenizer
clear_gpu()
print("Done! GPU memory released.")

Syncing to Google Drive...
KSAA 2026 - Sync Results to Google Drive
  Project: .

[1/3] Syncing outputs...
Transferred:   	   66.261 KiB / 474.738 KiB, 14%, 0 B/s, ETA -
Checks:                40 / 40, 100%
Transferred:            0 / 7, 0%
Elapsed time:         0.9s
Transferring:
 *                byt5_glonor_ft_checkpoint.json:  0% /63.964Ki, 0/s, -
 *           byt5_glonor_ft_dev_predictions.json:  0% /61.233Ki, 0/s, -
 *           byt5_glonor_ft_test_checkpoint.json:  0% /59.167Ki, 0/s, -
 *      whisper_tashkeel_ft_test_checkpoint.json:  0% /81.979Ki, 0/s, -
 *      whisper_tashkeel_ft_dev_predictions.json:  0% /63.530Ki, 0/s, -
 *     whisper_tashkeel_ft_test_predictions.json:  0% /78.604Ki, 0/s, -
 *           whisper_tashkeel_ft_checkpoint.json:100% /66.261Ki, 0/s, -Transferred:   	  474.738 KiB / 474.738 KiB, 100%, 0 B/s, ETA -
Checks:                43 / 43, 100%
Transferred:            0 / 7, 0%
Elapsed time:         1.4s
Transferring:
 *                byt5_glonor_ft_checkp